In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:


from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.signal import lfilter


# ---------------------------------------------------------
# TEXT TO BITS
# ---------------------------------------------------------

def text_to_bits(text):

    return ''.join(format(ord(c), '08b') for c in text)


# ---------------------------------------------------------
# BITS TO TEXT
# ---------------------------------------------------------

def bits_to_text(bits):

    chars = []

    for i in range(0, len(bits), 8):

        byte = bits[i:i+8]

        chars.append(chr(int(byte, 2)))

    return ''.join(chars)


# ---------------------------------------------------------
# EMBED MESSAGE
# ---------------------------------------------------------

def embed_message(
    input_wav,
    output_wav,
    text,
    d0=150,
    d1=200,
    alpha=0.5,
    frame_len=8192
):

    # Load audio
    audio, sr = sf.read(input_wav)

    # Stereo → Mono
    if audio.ndim > 1:
        audio = audio[:, 0]

    audio = audio.astype(np.float64)

    # Convert text to bits
    bits = text_to_bits(text)

    required_samples = len(bits) * frame_len

    # Capacity check
    if len(audio) < required_samples:
        raise ValueError("Audio too short!")

    stego = np.copy(audio)

    # Embed each bit
    for i, bit in enumerate(bits):

        start = i * frame_len
        end = start + frame_len

        frame = audio[start:end]

        # Bit 0
        if bit == '0':

            kernel = np.zeros(d0 + 1)

            kernel[-1] = alpha

        # Bit 1
        else:

            kernel = np.zeros(d1 + 1)

            kernel[-1] = alpha

        echoed = lfilter(kernel, [1.0], frame)

        stego[start:end] = frame + echoed

    # Save stego audio
    sf.write(output_wav, stego, sr)

    print("\nMessage embedded successfully!")
    print("Output file:", output_wav)


# ---------------------------------------------------------
# EXTRACT MESSAGE
# ---------------------------------------------------------

def extract_message(
    input_wav,
    message_length,
    d0=150,
    d1=200,
    frame_len=8192
):

    # Load audio
    audio, sr = sf.read(input_wav)

    # Stereo → Mono
    if audio.ndim > 1:
        audio = audio[:, 0]

    num_bits = message_length * 8

    retrieved_bits = []

    # Extract each bit
    for i in range(num_bits):

        start = i * frame_len
        end = start + frame_len

        frame = audio[start:end]

        spectrum = np.fft.fft(frame)

        log_mag = np.log(np.abs(spectrum) + 1e-10)

        cepstrum = np.fft.ifft(log_mag).real

        # Compare cepstrum peaks
        if cepstrum[d0] > cepstrum[d1]:

            retrieved_bits.append('0')

        else:

            retrieved_bits.append('1')

    retrieved_bits = ''.join(retrieved_bits)

    return bits_to_text(retrieved_bits)


# ---------------------------------------------------------
# MAIN PROGRAM
# ---------------------------------------------------------

if __name__ == "__main__":

    # WAV filename input
    cover = input("Enter WAV filename: ")

    # Secret message input
    message = input("Enter secret message: ")

    # Output filename
    stego = "stego_" + cover

    # Check file exists
    if Path(cover).exists():

        # Embed message
        embed_message(cover, stego, message)

        # Extract message
        recovered = extract_message(
            stego,
            len(message)
        )

        print("\nRecovered Message:", recovered)

    else:

        print("\nFile not found!")

In [ ]:
files.download("")